In [1]:
import pandas as pd
from sqlalchemy import create_engine

# Correct connection string
engine = create_engine("postgresql+psycopg2://postgres:Zxcvbnm,./123@localhost:5432/attrition")

# Load CSV
attrition_raw_data = pd.read_csv(r"C:\Users\HP\Desktop\HR data\WA_Fn-UseC_-HR-Employee-Attrition.csv")

# Upload to PostgreSQL
attrition_raw_data.to_sql("employee_attrition", engine, if_exists="replace", index=False)

print("Table loaded successfully!")
print(attrition_raw_data.shape)
print(attrition_raw_data.columns.tolist())

Table loaded successfully!
(1470, 35)
['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


In [2]:
# Overall attrition rate
query1 = """
SELECT "Attrition",
    COUNT(*) AS total_employees,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM employee_attrition
GROUP BY "Attrition";
"""
overall = pd.read_sql(query1, engine)
print(overall)

# Overall attrition rate by department
query2 = """
SELECT "Department", COUNT(*) as Total_Employees,
SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) AS Left_Employees,
	ROUND(SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) * 100 / COUNT(*), 2) AS attrition_rate
FROM employee_attrition
GROUP BY "Department"
ORDER BY attrition_rate DESC;
"""
department_attrition = pd.read_sql(query2, engine)
print(department_attrition)

# Overtime attrition rate
query3 = """
SELECT "OverTime", COUNT(*) as Total_Employees, SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) AS Left_Employees,
	ROUND(SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) * 100 / COUNT(*), 2) AS attrition_rate
FROM employee_attrition
GROUP BY "OverTime"
ORDER BY attrition_rate DESC;
"""
overtime_attrition  =pd.read_sql(query3, engine)
print(overtime_attrition)

  Attrition  total_employees  percentage
0        No             1233       83.88
1       Yes              237       16.12
               Department  total_employees  left_employees  attrition_rate
0                   Sales              446              92            20.0
1         Human Resources               63              12            19.0
2  Research & Development              961             133            13.0
  OverTime  total_employees  left_employees  attrition_rate
0      Yes              416             127            30.0
1       No             1054             110            10.0


In [3]:
# Job role attrition rate
query4 = """
SELECT "JobRole", COUNT(*) as Total_Employees, SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) AS Left_Employees,
	ROUND(SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) * 100 / COUNT(*), 2) AS attrition_rate
FROM employee_attrition
GROUP BY "JobRole"
ORDER BY attrition_rate DESC;
"""
Job_role_attrition = pd.read_sql(query4, engine)
print(Job_role_attrition)

# Gender attrition rate
query5 = """
SELECT "Gender", COUNT(*) as Total_Employees, SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) AS Left_Employees,
	ROUND(SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) * 100 / COUNT(*), 2) AS attrition_rate
FROM employee_attrition
GROUP BY "Gender"
ORDER BY attrition_rate DESC;
"""
gender_attrition = pd.read_sql(query5, engine)
print(gender_attrition)

# Marital status attrition rate
query6 = """
 SELECT "MaritalStatus", COUNT(*) as Total_Employees, SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) AS Left_Employees,
	ROUND(SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) * 100 / COUNT(*), 2) AS attrition_rate
FROM employee_attrition
GROUP BY "MaritalStatus"
ORDER BY attrition_rate DESC;
"""
marital_status_attrition = pd.read_sql(query6, engine)
print(marital_status_attrition)

                     JobRole  total_employees  left_employees  attrition_rate
0       Sales Representative               83              33            39.0
1      Laboratory Technician              259              62            23.0
2            Human Resources               52              12            23.0
3            Sales Executive              326              57            17.0
4         Research Scientist              292              47            16.0
5  Healthcare Representative              131               9             6.0
6     Manufacturing Director              145              10             6.0
7                    Manager              102               5             4.0
8          Research Director               80               2             2.0
   Gender  total_employees  left_employees  attrition_rate
0    Male              882             150            17.0
1  Female              588              87            14.0
  MaritalStatus  total_employees  left_empl

In [4]:
# Age attrition rate
query7 = """
SELECT 
    CASE 
        WHEN "Age" BETWEEN 18 AND 25 THEN '18-25'
        WHEN "Age" BETWEEN 26 AND 35 THEN '26-35'
        WHEN "Age" BETWEEN 36 AND 45 THEN '36-45'
        WHEN "Age" BETWEEN 46 AND 55 THEN '46-55'
        ELSE '55+'
    END AS age_group,
    COUNT(*) AS total_employees,
    SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) AS left_employees,
    ROUND(SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS attrition_rate
FROM employee_attrition
GROUP BY age_group
ORDER BY attrition_rate DESC;
"""
age_attrition = pd.read_sql(query7, engine)
print(age_attrition)


# Years at Company
query8 = """
SELECT 
    CASE 
        WHEN "YearsAtCompany" BETWEEN 0 AND 2 THEN '0-2 years'
        WHEN "YearsAtCompany" BETWEEN 3 AND 5 THEN '3-5 years'
        WHEN "YearsAtCompany" BETWEEN 6 AND 10 THEN '6-10 years'
        WHEN "YearsAtCompany" BETWEEN 11 AND 20 THEN '11-20 years'
        ELSE '20+ years'
    END AS years_at_company,
    COUNT(*) AS total_employees,
    SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) AS left_employees,
    ROUND(SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS attrition_rate
FROM employee_attrition
GROUP BY years_at_company
ORDER BY attrition_rate DESC;
"""
years_at_company_attrition = pd.read_sql(query8, engine)
print(years_at_company_attrition)

# Monthly Income
query9 = """
SELECT 
    CASE 
        WHEN "MonthlyIncome" < 3000 THEN 'Below 3k'
        WHEN "MonthlyIncome" BETWEEN 3000 AND 5000 THEN '3k-5k'
        WHEN "MonthlyIncome" BETWEEN 5001 AND 8000 THEN '5k-8k'
        WHEN "MonthlyIncome" BETWEEN 8001 AND 12000 THEN '8k-12k'
        ELSE 'Above 12k'
    END AS salary_group,
    COUNT(*) AS total_employees,
    SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) AS left_employees,
    ROUND(SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS attrition_rate
FROM employee_attrition
GROUP BY salary_group
ORDER BY attrition_rate DESC;
"""
monthly_income_attrition = pd.read_sql(query9, engine)
print(monthly_income_attrition)

  age_group  total_employees  left_employees  attrition_rate
0     18-25              123              44           35.77
1     26-35              606             116           19.14
2       55+               47               8           17.02
3     46-55              226              26           11.50
4     36-45              468              43            9.19
  years_at_company  total_employees  left_employees  attrition_rate
0        0-2 years              342             102           29.82
1        3-5 years              434              60           13.82
2       6-10 years              448              55           12.28
3        20+ years               66               8           12.12
4      11-20 years              180              12            6.67
  salary_group  total_employees  left_employees  attrition_rate
0     Below 3k              395             113           28.61
1       8k-12k              186              29           15.59
2        3k-5k              354   

In [5]:
# Job Satisfaction attrition rate
query10 = """
SELECT CASE 
		WHEN "JobSatisfaction" Between 1 AND 2 THEN 'Low'
		WHEN "JobSatisfaction" Between 2 AND 3 THEN 'Average'
		ELSE 'High'
	END AS Satisfactory_level,
	COUNT(*) AS total_employees,
SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) AS Left_Employees,
	ROUND(SUM(CASE WHEN "Attrition" = 'Yes' THEN 1 ELSE 0 END) * 100 / COUNT(*), 2) AS attrition_rate
FROM employee_attrition
GROUP BY Satisfactory_level
ORDER BY attrition_rate DESC;
"""
job_satisfaction_attrition = pd.read_sql(query10, engine)
print(job_satisfaction_attrition)

  satisfactory_level  total_employees  left_employees  attrition_rate
0                Low              569             112            19.0
1            Average              442              73            16.0
2               High              459              52            11.0
